# X5: Benchmarking & Systematic Evaluation

**Professional Practice Series - Part 5**

---

## 📋 Overview

This notebook covers comprehensive benchmarking and evaluation practices for RL:

**Topics:**
1. Standard benchmarking suites (Atari, MuJoCo, DeepMind Control)
2. Evaluation protocols and statistical testing
3. Performance metrics beyond reward
4. Sample efficiency analysis
5. Reproducibility and random seeds
6. Creating custom benchmarks
7. Continuous integration for RL
8. Reporting best practices

**Why This Matters:**
- Published RL results are often not reproducible
- Proper evaluation is critical for scientific rigor
- Benchmarks allow comparison across methods
- Industry needs reliable performance metrics

**Pre-requisites:** Lessons 0-9, X3 (Debugging)

---

## 1. Standard RL Benchmarks

### The Challenge of RL Benchmarking

Unlike supervised learning (MNIST, ImageNet), RL benchmarking is harder:
- **High variance** in training (even with same hyperparameters)
- **Sensitivity** to random seeds
- **Non-stationarity** makes comparison difficult
- **Computational cost** limits number of trials

### Major Benchmark Suites

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import json
from pathlib import Path
from scipy import stats
from collections import defaultdict

# Standard benchmark suites
BENCHMARK_SUITES = {
    "classic_control": [
        "CartPole-v1",  # Solved: 475
        "Acrobot-v1",  # Solved: -100
        "MountainCar-v0",  # Solved: -110
        "Pendulum-v1",  # Continuous control
    ],
    "box2d": [
        "LunarLander-v2",  # Solved: 200
        "LunarLanderContinuous-v2",  # Continuous
        "BipedalWalker-v3",  # Solved: 300
        "CarRacing-v2",  # Vision-based
    ],
    "atari": [
        "ALE/Pong-v5",
        "ALE/Breakout-v5",
        "ALE/SpaceInvaders-v5",
        "ALE/Qbert-v5",
        # Full Atari suite: 57 games
    ],
    "mujoco": [
        "HalfCheetah-v4",
        "Hopper-v4",
        "Walker2d-v4",
        "Ant-v4",
        "Humanoid-v4",
    ],
}

# Print benchmark info
for suite_name, envs in BENCHMARK_SUITES.items():
    print(f"\n{suite_name.upper()} ({len(envs)} envs):")
    for env_name in envs[:3]:  # Show first 3
        print(f"  - {env_name}")
    if len(envs) > 3:
        print(f"  ... and {len(envs) - 3} more")

## 2. Proper Evaluation Protocol

### Common Mistakes
❌ Evaluating on training environment (data leakage)  
❌ Single random seed  
❌ Reporting max instead of mean  
❌ No confidence intervals  
❌ Tuning on test set  

### Correct Protocol
✅ Separate evaluation environment  
✅ Multiple random seeds (≥5, ideally ≥10)  
✅ Report mean ± std  
✅ Statistical significance testing  
✅ Hold-out hyperparameter tuning  

In [ ]:
class RLEvaluator:
    """Rigorous RL evaluation with statistical testing."""

    def __init__(
        self,
        env_name: str,
        n_eval_episodes: int = 100,
        n_seeds: int = 5,
        confidence_level: float = 0.95,
    ):
        self.env_name = env_name
        self.n_eval_episodes = n_eval_episodes
        self.n_seeds = n_seeds
        self.confidence_level = confidence_level

        self.results = defaultdict(list)

    def evaluate_agent(
        self, agent, seed: int, deterministic: bool = True
    ) -> Dict:
        """Evaluate single agent with one seed."""
        env = gym.make(self.env_name)
        env.reset(seed=seed)

        episode_rewards = []
        episode_lengths = []
        success_rate = []

        for episode in range(self.n_eval_episodes):
            state, _ = env.reset(seed=seed + episode)
            episode_reward = 0
            episode_length = 0
            done = False

            while not done:
                # Get action (depends on your agent interface)
                action = agent.select_action(state, deterministic=deterministic)

                state, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated

                episode_reward += reward
                episode_length += 1

            episode_rewards.append(episode_reward)
            episode_lengths.append(episode_length)

            # Check success (if environment provides it)
            if "is_success" in info:
                success_rate.append(float(info["is_success"]))

        env.close()

        return {
            "mean_reward": np.mean(episode_rewards),
            "std_reward": np.std(episode_rewards),
            "median_reward": np.median(episode_rewards),
            "mean_length": np.mean(episode_lengths),
            "success_rate": np.mean(success_rate) if success_rate else None,
            "rewards": episode_rewards,
        }

    def evaluate_multiple_seeds(
        self, agent_fn, seeds: List[int] = None
    ) -> Dict:
        """
        Evaluate agent across multiple random seeds.

        Args:
            agent_fn: Function that returns fresh agent given seed
            seeds: List of random seeds to use

        Returns:
            Aggregated statistics with confidence intervals
        """
        if seeds is None:
            seeds = list(range(self.n_seeds))

        all_mean_rewards = []
        all_episode_rewards = []

        for seed in seeds:
            # Train agent with this seed
            agent = agent_fn(seed)

            # Evaluate
            results = self.evaluate_agent(agent, seed)
            all_mean_rewards.append(results["mean_reward"])
            all_episode_rewards.extend(results["rewards"])

        # Compute statistics
        mean_across_seeds = np.mean(all_mean_rewards)
        std_across_seeds = np.std(all_mean_rewards)

        # Confidence interval
        ci = stats.t.interval(
            self.confidence_level,
            len(all_mean_rewards) - 1,
            loc=mean_across_seeds,
            scale=stats.sem(all_mean_rewards),
        )

        return {
            "mean": mean_across_seeds,
            "std": std_across_seeds,
            "median": np.median(all_mean_rewards),
            "min": np.min(all_mean_rewards),
            "max": np.max(all_mean_rewards),
            "confidence_interval": ci,
            "all_seeds": all_mean_rewards,
        }

    def compare_algorithms(
        self, algo1_results: List[float], algo2_results: List[float]
    ) -> Dict:
        """
        Statistical comparison of two algorithms.

        Uses Mann-Whitney U test (non-parametric).
        """
        # Mann-Whitney U test (doesn't assume normality)
        u_statistic, p_value = stats.mannwhitneyu(
            algo1_results, algo2_results, alternative="two-sided"
        )

        # Effect size (Cohen's d)
        mean_diff = np.mean(algo1_results) - np.mean(algo2_results)
        pooled_std = np.sqrt(
            (np.var(algo1_results) + np.var(algo2_results)) / 2
        )
        cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0

        return {
            "p_value": p_value,
            "significant": p_value < 0.05,
            "u_statistic": u_statistic,
            "cohens_d": cohens_d,
            "mean_difference": mean_diff,
            "interpretation": (
                "Algorithm 1 significantly better"
                if p_value < 0.05 and mean_diff > 0
                else "Algorithm 2 significantly better"
                if p_value < 0.05 and mean_diff < 0
                else "No significant difference"
            ),
        }


# Example usage
print("Evaluation protocol ready!")
print("\nProper evaluation includes:")
print("  1. Multiple random seeds (≥5)")
print("  2. Separate evaluation episodes (≥100)")
print("  3. Mean ± Std reporting")
print("  4. Confidence intervals")
print("  5. Statistical significance testing")

## 3. Performance Metrics

**Don't just report final reward!**

### Comprehensive Metrics

In [ ]:
class ComprehensiveMetrics:
    """Track comprehensive RL performance metrics."""

    @staticmethod
    def compute_sample_efficiency(
        rewards_over_time: List[float], target_reward: float
    ) -> int:
        """Compute sample efficiency: steps to reach target."""
        for step, reward in enumerate(rewards_over_time):
            if reward >= target_reward:
                return step
        return len(rewards_over_time)  # Never reached

    @staticmethod
    def compute_auc(rewards: List[float]) -> float:
        """Area Under Curve - total cumulative reward."""
        return np.trapz(rewards)

    @staticmethod
    def compute_final_performance(
        rewards: List[float], window: int = 100
    ) -> float:
        """Average reward over final window."""
        return np.mean(rewards[-window:])

    @staticmethod
    def compute_stability(rewards: List[float], window: int = 100) -> float:
        """Variance in final performance (lower = more stable)."""
        return np.std(rewards[-window:])

    @staticmethod
    def compute_forgetting(rewards: List[float], window: int = 100) -> float:
        """Peak performance - final performance (detect catastrophic forgetting)."""
        peak = np.max(rewards)
        final = np.mean(rewards[-window:])
        return peak - final

    @staticmethod
    def compute_all_metrics(rewards: List[float]) -> Dict:
        """Compute full suite of metrics."""
        return {
            "final_performance": ComprehensiveMetrics.compute_final_performance(
                rewards
            ),
            "peak_performance": np.max(rewards),
            "auc": ComprehensiveMetrics.compute_auc(rewards),
            "stability": ComprehensiveMetrics.compute_stability(rewards),
            "forgetting": ComprehensiveMetrics.compute_forgetting(rewards),
            "mean": np.mean(rewards),
            "std": np.std(rewards),
            "median": np.median(rewards),
        }


# Example
example_rewards = [10 + i * 2 + np.random.randn() * 5 for i in range(100)]
metrics = ComprehensiveMetrics.compute_all_metrics(example_rewards)

print("\nComprehensive Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.2f}")

## 4. Reproducibility

### The Reproducibility Crisis in RL

Many published RL results cannot be reproduced ([Henderson et al., 2018](https://arxiv.org/abs/1709.06560)).

**Common issues:**
- Missing hyperparameters
- Unreported random seeds
- Different library versions
- Code not released
- Cherry-picked results

In [ ]:
import hashlib
import sys


class ReproducibilityTracker:
    """Track all information needed for reproducibility."""

    @staticmethod
    def get_environment_info() -> Dict:
        """Get complete environment information."""
        import platform
        import torch

        return {
            "python_version": sys.version,
            "platform": platform.platform(),
            "numpy_version": np.__version__,
            "torch_version": torch.__version__,
            "gymnasium_version": gym.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        }

    @staticmethod
    def set_global_seeds(seed: int):
        """Set all random seeds for reproducibility."""
        import random
        import torch

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            # Make CUDA deterministic (may slow down)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    @staticmethod
    def save_experiment_config(
        config: Dict, filepath: str, include_env: bool = True
    ):
        """Save complete experiment configuration."""
        if include_env:
            config["environment"] = ReproducibilityTracker.get_environment_info()

        with open(filepath, "w") as f:
            json.dump(config, f, indent=2, default=str)

        print(f"✓ Configuration saved to {filepath}")

    @staticmethod
    def compute_results_hash(results: Dict) -> str:
        """Compute hash of results for verification."""
        results_str = json.dumps(results, sort_keys=True, default=str)
        return hashlib.sha256(results_str.encode()).hexdigest()[:16]


# Example: Save reproducible config
config = {
    "algorithm": "PPO",
    "env_name": "CartPole-v1",
    "seed": 42,
    "hyperparameters": {
        "lr": 3e-4,
        "gamma": 0.99,
        "clip_epsilon": 0.2,
    },
    "training": {"n_steps": 2048, "n_iterations": 500},
}

print("\nReproducibility Checklist:")
print("  ✓ All hyperparameters saved")
print("  ✓ Random seeds fixed")
print("  ✓ Library versions recorded")
print("  ✓ Code released (ideally)")
print("  ✓ Full evaluation protocol documented")

## 5. Visualization Best Practices

In [ ]:
def plot_training_curves(
    results: Dict[str, List[float]], smooth_window: int = 10
):
    """
    Plot training curves with best practices.

    Best practices:
    - Show both raw and smoothed curves
    - Include confidence intervals
    - Label axes clearly
    - Use colorblind-friendly palette
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = plt.cm.Set2(range(len(results)))

    for i, (name, rewards) in enumerate(results.items()):
        # Raw curve (transparent)
        ax.plot(rewards, alpha=0.2, color=colors[i])

        # Smoothed curve
        if len(rewards) >= smooth_window:
            smoothed = np.convolve(
                rewards, np.ones(smooth_window) / smooth_window, mode="valid"
            )
            ax.plot(
                range(smooth_window - 1, len(rewards)),
                smoothed,
                label=name,
                color=colors[i],
                linewidth=2,
            )

    ax.set_xlabel("Episode", fontsize=12)
    ax.set_ylabel("Reward", fontsize=12)
    ax.set_title("Training Performance Comparison", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


def plot_confidence_intervals(algorithm_results: Dict[str, List[List[float]]]):
    """
    Plot with confidence intervals across multiple seeds.

    Args:
        algorithm_results: {algo_name: [seed1_rewards, seed2_rewards, ...]}
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    for algo_name, seeds_rewards in algorithm_results.items():
        # Convert to array: (n_seeds, n_steps)
        rewards_array = np.array(seeds_rewards)

        # Compute mean and std across seeds
        mean_rewards = np.mean(rewards_array, axis=0)
        std_rewards = np.std(rewards_array, axis=0)

        steps = np.arange(len(mean_rewards))

        # Plot mean
        ax.plot(steps, mean_rewards, label=algo_name, linewidth=2)

        # Plot confidence interval
        ax.fill_between(
            steps,
            mean_rewards - std_rewards,
            mean_rewards + std_rewards,
            alpha=0.2,
        )

    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward (mean ± std)")
    ax.set_title("Performance with Confidence Intervals")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


print("Visualization utilities ready!")

## 6. Sample Efficiency Analysis

**Sample efficiency** = How quickly does the algorithm learn?

Critical for real-world applications (robotics, expensive simulations).

In [ ]:
def compute_sample_efficiency_curve(
    rewards: List[float], target_reward: float, window: int = 100
) -> Tuple[List[int], List[float]]:
    """
    Compute learning curve: performance vs. environment steps.

    Returns:
        steps, rewards at those steps
    """
    steps = []
    performance = []

    for i in range(0, len(rewards), window):
        steps.append(i)
        performance.append(np.mean(rewards[max(0, i - window) : i]))

    return steps, performance


def compare_sample_efficiency(
    results: Dict[str, List[float]], target_reward: float
):
    """Compare sample efficiency across algorithms."""
    print("\nSample Efficiency Comparison:")
    print(f"Target reward: {target_reward}")
    print("-" * 50)

    efficiency = {}

    for name, rewards in results.items():
        steps_to_solve = ComprehensiveMetrics.compute_sample_efficiency(
            rewards, target_reward
        )

        efficiency[name] = steps_to_solve
        status = "✓ Solved" if steps_to_solve < len(rewards) else "✗ Not solved"
        print(f"{name:20s}: {steps_to_solve:6d} steps  {status}")

    # Rank by efficiency
    ranked = sorted(efficiency.items(), key=lambda x: x[1])
    print(f"\nMost efficient: {ranked[0][0]} ({ranked[0][1]} steps)")

    return efficiency


print("Sample efficiency analysis ready!")

## 7. Benchmark Report Template

In [ ]:
class BenchmarkReport:
    """Generate comprehensive benchmark report."""

    def __init__(self, experiment_name: str):
        self.experiment_name = experiment_name
        self.results = {}

    def add_algorithm(
        self, name: str, rewards: List[List[float]], config: Dict
    ):
        """Add algorithm results."""
        self.results[name] = {"rewards": rewards, "config": config}

    def generate_report(self, output_file: str = None) -> str:
        """Generate markdown report."""
        report = []
        report.append(f"# Benchmark Report: {self.experiment_name}\n")

        report.append("## Executive Summary\n")
        report.append(f"- **Date**: {Path('.').stat().st_mtime}")
        report.append(f"- **Algorithms tested**: {len(self.results)}")
        report.append("")

        report.append("## Results\n")
        report.append("| Algorithm | Mean Reward | Std | CI (95%) |")
        report.append("|-----------|-------------|-----|----------|")

        for name, data in self.results.items():
            rewards = data["rewards"]
            mean_rewards = [np.mean(r) for r in rewards]
            overall_mean = np.mean(mean_rewards)
            overall_std = np.std(mean_rewards)

            ci = stats.t.interval(
                0.95, len(mean_rewards) - 1, loc=overall_mean, scale=stats.sem(mean_rewards)
            )

            report.append(
                f"| {name} | {overall_mean:.2f} | {overall_std:.2f} | [{ci[0]:.2f}, {ci[1]:.2f}] |"
            )

        report.append("")
        report.append("## Reproducibility\n")
        report.append("Environment info:")
        env_info = ReproducibilityTracker.get_environment_info()
        for key, value in env_info.items():
            report.append(f"- {key}: {value}")

        report_text = "\n".join(report)

        if output_file:
            with open(output_file, "w") as f:
                f.write(report_text)
            print(f"✓ Report saved to {output_file}")

        return report_text


print("\nBenchmark report generator ready!")
print("Use to create publication-quality reports.")

## Summary

### Benchmarking Checklist

✅ **Environment**
- [ ] Use standard benchmark (Atari, MuJoCo, etc.)
- [ ] Separate train/eval environments
- [ ] Document environment version

✅ **Evaluation Protocol**
- [ ] Multiple random seeds (≥5)
- [ ] Sufficient evaluation episodes (≥100)
- [ ] Report mean ± std
- [ ] Compute confidence intervals
- [ ] Statistical significance testing

✅ **Metrics**
- [ ] Final performance
- [ ] Sample efficiency
- [ ] Stability
- [ ] Peak performance
- [ ] Area under curve

✅ **Reproducibility**
- [ ] All hyperparameters saved
- [ ] Random seeds documented
- [ ] Library versions recorded
- [ ] Code released (ideally)
- [ ] Full protocol documented

✅ **Reporting**
- [ ] Clear visualizations
- [ ] Confidence intervals shown
- [ ] Comparison with baselines
- [ ] Limitations discussed

---

**Key Takeaway**: Rigorous benchmarking is essential for scientific RL research and production deployment.

**Next Steps**:
- Run your algorithm on standard benchmarks
- Compare with published baselines
- Report full statistics
- Make results reproducible

---

*See Lesson X6 for advanced topics and future directions*
*See Examples for benchmark implementations*